# Codebook Ablation Study — OsrSAF_TriNet

Trains and evaluates 4 variants of the OSR head, isolating the contribution of each codebook:

| Variant | Cosine codebook | Hamming codebook | Calibrator inputs |
|---|---|---|---|
| `neither` | masked | masked | backbone-only floor (3 features) |
| `cosine`  | active | masked | 6 features |
| `hamming` | masked | active | 5 features |
| `full`    | active | active | full 8 features (control) |

## Steps
1. Mount Drive and set paths
2. Verify model files are in place
3. Run training (all 4 variants, skips already-trained checkpoints)
4. Run evaluation across all 13 SNR points (saves t-SNE + confusion matrix + JSON per point)
5. (Optional) Sanity check: parameter counts and mask layouts per variant

In [1]:
# ── Step 1: Mount Drive and set paths ───────────────────────────────
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My Laptop/thesis_project


In [3]:
# ── Step 2: Verify required files are in place ────────────────────────
# Required:
#   python/src/models/ablation_osr_saf_trinet.py
#   scripts/run_codebook_ablation.py
#   scripts/run_codebook_ablation_eval.py
#   artifacts/checkpoints/asymmetric_trinet_seed42_n2500.pt   (closed-set backbone)

import shutil

expected = [
    PROJECT_ROOT / 'python/src/models/ablation_osr_saf_trinet.py',
    PROJECT_ROOT / 'scripts/run_codebook_ablation.py',
    PROJECT_ROOT / 'scripts/run_codebook_ablation_eval.py',
    PROJECT_ROOT / 'artifacts/checkpoints/asymmetric_trinet_seed42_n2500.pt',
]

all_ok = True
for p in expected:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p}')
    if not p.exists():
        all_ok = False

# Make sure AblationOsrSAF_TriNet is exported from the models package
init_path = PROJECT_ROOT / 'python/src/models/__init__.py'
init_text = init_path.read_text() if init_path.exists() else ''
if 'AblationOsrSAF_TriNet' not in init_text:
    print('\n  [INFO] AblationOsrSAF_TriNet not yet exported from python/src/models/__init__.py')
    print('  Add this line:')
    print("      from .ablation_osr_saf_trinet import AblationOsrSAF_TriNet")
    print('  (Not strictly required — the scripts import directly from the module path — but recommended.)')

print('\nAll required files present.' if all_ok else '\nSome files missing — fix above before continuing.')

  [OK] /content/drive/Othercomputers/My Laptop/thesis_project/python/src/models/ablation_osr_saf_trinet.py
  [OK] /content/drive/Othercomputers/My Laptop/thesis_project/scripts/run_codebook_ablation.py
  [OK] /content/drive/Othercomputers/My Laptop/thesis_project/scripts/run_codebook_ablation_eval.py
  [OK] /content/drive/Othercomputers/My Laptop/thesis_project/artifacts/checkpoints/asymmetric_trinet_seed42_n2500.pt

All required files present.


In [4]:
# ── Step 3: Run Training ────────────────────────────────────────────
# This will sequentially train all 4 codebook ablation variants.
# Phase 1 fill (15 epochs) + Phase 2 calibrator (50 epochs).
# Already-trained variants are skipped automatically.

%run scripts/run_codebook_ablation.py

Device: cuda
Seed: 42 | n_per_class: 2500 | spec: v2

Loading OSR datasets...
[load_osr_datasets] proxy unknowns: 5000 samples (train 4000 / val 1000)
[load_osr_datasets] test  unknowns: 2000 samples (held out)

  Training: Neither codebook (backbone-only floor)
  Disabled codebooks: ['cosine', 'hamming']
[OsrSAF_TriNet] Loaded backbone from /content/drive/Othercomputers/My Laptop/thesis_project/artifacts/checkpoints/asymmetric_trinet_seed42_n2500.pt

  [Stage 2.A] Populating codebooks over 15 epochs (frozen backbone)
    Fill epoch 01/15 | init=100% | spread=0.0500 | updates/centroid=378.6
    Fill epoch 05/15 | init=100% | spread=0.0841 | updates/centroid=1889.5
    Fill epoch 10/15 | init=100% | spread=0.0719 | updates/centroid=3777.1
    Fill epoch 15/15 | init=100% | spread=0.0697 | updates/centroid=5665.4

  [Stage 2.B] Training calibrator on proxy unknowns
  Ep    | Loss    | KnAcc  | AUROC  | Recall | FPR    | Thr   
  -----------------------------------------------------------

In [ ]:
# ── Step 4: Run Evaluation across all 13 SNR points ───────────────────────
# For every (variant, SNR) pair this saves:
#   reports/figures/codebook_ablation/<variant>/snr_<snr>dB/osr_confusion_matrix.png
#   reports/figures/codebook_ablation/<variant>/snr_<snr>dB/osr_tsne_embedding.png
#   reports/figures/codebook_ablation/<variant>/snr_<snr>dB/osr_per_class_accuracy.json
# Plus a unified results JSON and LaTeX tables in:
#   artifacts/logs/codebook_ablation/

%run scripts/run_codebook_ablation_eval.py

In [ ]:
# ── Optional: sanity check ── parameter counts and mask layouts ───────────────
# Useful to confirm all variants are architecturally identical and that the
# masks zero out exactly the calibrator inputs we expect.

from python.src.models.ablation_osr_saf_trinet import AblationOsrSAF_TriNet

variants = [
    ('neither', {'cosine', 'hamming'}, 'Neither codebook'),
    ('cosine',  {'hamming'},           'Cosine only'),
    ('hamming', {'cosine'},            'Hamming only'),
    ('full',    set(),                 'Full (both)'),
]

feature_names = [
    'code_dist', 'unc', 'emb_norm', 'runner_up_dist',
    'margin_codebook', 'logit_margin', 'hamming_dist', 'hamming_margin',
]

print(f"{'Variant':<18} {'Calib params':>14}  Active calibrator inputs")
print('-' * 90)
for tag, disabled, label in variants:
    m = AblationOsrSAF_TriNet(
        num_classes=10,
        disabled_codebooks=disabled,
        use_pretrained=False,
    )
    n_params = sum(p.numel() for p in m.score_calibrator.parameters() if p.requires_grad)
    mask = m.score_calibrator.mask.cpu().numpy()
    active = [feature_names[i] for i, v in enumerate(mask) if v > 0.5]
    print(f'{label:<18} {n_params:>14,}  {active}')

In [ ]:
# ── Optional: Backup results to Drive (only if you copied the project to local Colab disk) ──
# DRIVE_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')
# !mkdir -p "{DRIVE_ROOT}/reports"
# !mkdir -p "{DRIVE_ROOT}/artifacts"
# !cp -r -u "{PROJECT_ROOT}/reports/."   "{DRIVE_ROOT}/reports/"
# !cp -r -u "{PROJECT_ROOT}/artifacts/." "{DRIVE_ROOT}/artifacts/"
# print('Backup complete. Safe to close Colab.')